# Which model captures *my* biology?

**What you'll learn.** Instead of guessing which foundation model to use,
embpy lets you *measure* it: `tl.benchmark_embeddings` trains simple probes on
an embedding and scores how well it predicts a target you care about (a
measured property, a viability score, an expression profile). Run it across
several models and you get a ranking on **your** data.

This is embpy's core value proposition — turning "which model is best?" into a
number instead of an opinion.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd

from embpy import BioEmbedder, tl

embedder = BioEmbedder(device="auto", organism="human")

## Your data: entities plus a target to predict

You bring the identifiers and a numeric label per entity. Here we use a small
set of genes and an illustrative target column — swap in your real measurement
(an IC50, a dependency score, a phenotype readout) and everything else stays
the same.

> The point of a benchmark is the *comparison between models*, so a shared,
> honest target matters more than its source. On a real task, use a real label.

In [ ]:
genes = ["TP53", "EGFR", "MYC", "BRCA1", "JUN", "STAT1", "IRF1", "CDK1",
         "GATA3", "FOXP3", "CD8A", "IL2", "CCND1", "RB1", "PTEN", "AKT1"]
rng = np.random.default_rng(0)

adata = ad.AnnData(
    X=np.zeros((len(genes), 1), dtype=np.float32),
    obs=pd.DataFrame(
        {"symbol": genes, "target": rng.normal(size=len(genes)).astype(np.float32)},
        index=genes,
    ),
)

## Benchmark several models

The workflow is *embed, then score*: compute each model's embedding into
`.obsm`, then hand that key to `benchmark_embeddings`, which trains a set of
regressors (linear, ridge, kNN, random forest) to predict the target and
returns a metrics table — one row per regressor. Loop over the models you're
weighing and keep each one's best score.

Here we compare two prior-knowledge gene embeddings; add sequence models like
`esm2_8M` or `hyenadna_small_32k` (they download once) to weigh across model
*types*.

In [ ]:
candidates = ["genept", "gene2vec"]

leaderboard = {}
for model in candidates:
    scored = embedder.embed(
        adata.copy(), entity_type="gene", id_type="symbol", obs_column="symbol",
        model=model, output="anndata", key="X_emb",
    )
    res = tl.benchmark_embeddings(
        scored,
        perturbation_column="symbol",
        perturbation_type="genetic",
        target="target",
        obsm_key="X_emb",          # score the embedding we just computed
        models=["ridge", "knn"],
        mode="quick",
    )
    leaderboard[model] = res["r2"].max()   # best regressor for this embedding

ranking = pd.Series(leaderboard, name="best_r2").sort_values(ascending=False)

## Read the ranking

Higher R² means the embedding carries more information about your target. The
winner here is the model you'd take forward — for *this* task, on *your* data.

In [ ]:
print(ranking.to_string())

## Always check the baseline

A leaderboard is only meaningful against a floor. `benchmark_embeddings`
already trains a plain **linear** probe alongside the others — if a foundation
model can't beat linear-on-the-raw-embedding, its extra capacity bought you
nothing for this task. Two honest results from the perturbation literature are
worth remembering:

- **Prior-knowledge** embeddings (like `genept`) often beat sequence models on
  gene-level tasks — capability, not size, is what wins.
- For a serious comparison, use `mode="rigorous"` (5-fold CV + hyper-parameter
  search) and a **leave-entity-out** split, so the score reflects generalisation
  rather than memorising which gene is which.

## Takeaway

You now have a repeatable way to pick a model with evidence instead of
reputation. Point `benchmark_embeddings` at your labelled data, read the
leaderboard, and check the baseline.

The full results table is also stored on the AnnData at
`adata.uns["benchmark_results"]` for the last model benchmarked.